<a href="https://colab.research.google.com/github/redddddsu/2048-Arduino/blob/main/Software_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing API/Library


In [2]:
!pip install snntorch

In [3]:
import torch
import torch.nn as nn
import snntorch as snn
import matplotlib.pyplot as plt
import os
import librosa
import numpy as np
from snntorch import surrogate
import tensorflow as tf
import pandas as pd
import tensorflow_hub as hub
from sklearn.metrics import confusion_matrix, classification_report
import yaml
import zipfile
import random



/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Dataset

In [ ]:
save_dir = "/content/drive/MyDrive/TUT_dataset/mixtures"
os.makedirs(save_dir, exist_ok=True)

base_name = "TUT-rare-sound-events-2017-development.mixture_data"

for i in [1, 2, 3, 4, 5, 6, 7, 8, 9]:

    filename = f"{base_name}.{i}.zip"
    zip_path = os.path.join(save_dir, filename)
    url = f"https://zenodo.org/records/401395/files/{filename}?download=1"

    if not os.path.exists(zip_path):
        !wget -c -O "{zip_path}" "{url}"
    extract_marker = os.path.join(save_dir, f".extracted_{i}")

    if not os.path.exists(extract_marker):
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(save_dir)

        with open(extract_marker, "w") as f:
            f.write("done")

Mounted at /content/drive
--2026-11-02 12:07:28--  https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.mixture_data.1.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.47.192, 188.185.45.63, 188.185.40.132, ...
Connecting to zenodo.org (zenodo.org)|188.185.47.192|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1072878357 (1023M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/TUT_dataset/mixtures/TUT-rare-sound-events-2017-development.mixture_data.1.zip’

/content/drive/MyDr 100%[===================>]   1023M  3.03MB/s    in 5m 36s  

2026-11-02 12:13:05 (3.05 MB/s) - ‘/content/drive/MyDrive/TUT_dataset/mixtures/TUT-rare-sound-events-2017-development.mixture_data.1.zip’ saved [1072878357/1072878357]

--2026-11-02 12:13:50--  https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.mixture_data.2.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.51.60, 137.138.121.78, 188.185.47.

In [36]:
dataset_dir = "/content/drive/MyDrive/TUT_dataset/mixtures"

recordings = []

for root, dirs, files in os.walk(dataset_dir):
    if "devtrain" not in root:
        continue

    for file in files:
        if file.endswith(".wav"):
            wav_path = os.path.join(root, file)
            recordings.append({"wav_path": wav_path})

print("Total devtrain recordings:", len(recordings))

Total devtrain recordings: 1491


In [37]:
metadata_files = []

for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file.endswith(".yaml"):
            metadata_files.append(os.path.join(root, file))

all_recipes = []

for event in ["babycry", "glassbreak", "gunshot"]:
    yaml_path = next(
        path for path in metadata_files
        if f"mixture_recipes_devtrain_{event}.yaml" in path
    )

    with open(yaml_path, "r") as f:
        all_recipes.extend(yaml.safe_load(f))

recipe_lookup = {
    recipe["mixture_audio_filename"]: recipe
    for recipe in all_recipes
}

print("Total recipes:", len(all_recipes))

Total recipes: 1500


In [38]:
random.seed(42)

selected_recordings = {
    "babycry": [],
    "glassbreak": [],
    "gunshot": [],
    "background_only": []
}

for recording in recordings:
    filename = os.path.basename(recording["wav_path"])
    recipe = recipe_lookup[filename]

    if recipe["event_present"]:
        event = recipe["event_class"]
    else:
        event = "background_only"

    selected_recordings[event].append(recording)

balanced_recordings = []

for event, items in selected_recordings.items():
    balanced_recordings.extend(random.sample(items, 200))

print("Selected recordings:", len(balanced_recordings))

Selected recordings: 800


# Mel Spectrogem

In [39]:
mel_data = {}

n_mels = 128
window_seconds = 0.064
hop_seconds = 0.032

for recording in balanced_recordings:
    wav_path = recording["wav_path"]
    audio, sr = librosa.load(
        wav_path,
        sr=None,
        mono=True
    )
    window_sample = round(sr * window_seconds)
    hop_sample = round(sr * hop_seconds)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=window_sample,
        win_length=window_sample,
        hop_length=hop_sample,
        n_mels=n_mels,
        fmin=0,
        fmax=sr / 2,
        power=2
    )

    log_mel = librosa.power_to_db(mel, ref=1.0, top_db=None)
    mel_data[wav_path] = {
        "log_mel": log_mel.astype(np.float32),
        "sr": sr,
        "duration": len(audio) / sr,
    }

print("Processed recordings:", len(mel_data))

Processed recordings: 800


In [40]:
import pickle

cache_path = "/content/drive/MyDrive/TUT_dataset/mel_data_v2.pkl"

with open(cache_path, "wb") as f:
    pickle.dump(mel_data, f)

In [41]:
mel_data = {}

with open(cache_path, "rb") as f:
    mel_data = pickle.load(f)

# Labeling Frames

In [42]:
event_to_label = {
    "babycry": 1,
    "glassbreak": 2,
    "gunshot": 3
}
for recording in recordings:
    wav_path = recording["wav_path"]
    filename = os.path.basename(wav_path)

    if wav_path not in mel_data:
        continue

    if filename not in recipe_lookup:
        continue

    recipe = recipe_lookup[filename]
    num_frames = mel_data[wav_path]["log_mel"].shape[1]
    labels = np.zeros(num_frames, dtype=np.int64)
    if recipe["event_present"]:
        start_time = recipe["event_start_in_mixture_seconds"]
        end_time = start_time + recipe["event_length_seconds"]
        start_frame = int(np.floor(start_time / hop_seconds))
        end_frame = int(np.ceil(end_time / hop_seconds))
        start_frame = max(0, min(start_frame, num_frames))
        end_frame = max(0, min(end_frame, num_frames))
        labels[start_frame:end_frame] = event_to_label[recipe["event_class"]]
    mel_data[wav_path]["labels"] = labels

print("Labeling complete.")

Labeling complete.


# SNN Frame


In [43]:
spike_grad = surrogate.fast_sigmoid()

class SNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(128, 128)
        self.lif1 = snn.Leaky(
            beta=0.8,
            threshold=1.0,
            spike_grad=spike_grad,
            reset_mechanism="subtract"
        )

        self.layer2 = nn.Linear(128, 1)
        self.lif2 = snn.Leaky(
          beta=0.8,
          threshold=1.0,
          spike_grad=spike_grad,
          reset_mechanism="subtract"
        )

    def forward(self, x):
        potential1 = self.lif1.init_leaky()
        potential2 = self.lif2.init_leaky()

        current1_all = self.layer1(x)
        spikes = []

        for t in range(x.shape[0]):
            spk1, potential1 = self.lif1(
                current1_all[t], potential1
            )
            current2 = self.layer2(spk1)
            spk2, potential2 = self.lif2(
                current2, potential2
            )
            spikes.append(spk2)

        return torch.stack(spikes)





# Van Rossum Loss

In [44]:
def van_rossum_loss(predict, target, dt=0.032, tau=0.1):
    alpha = torch.exp(torch.tensor(-dt / tau, device=predict.device))

    pred_filter = []
    target_filter = []

    pred_state = torch.zeros_like(predict[0])
    target_state = torch.zeros_like(target[0])

    for t in range(predict.shape[0]):
        pred_state = alpha * pred_state + predict[t]
        target_state = alpha * target_state + target[t]

        pred_filter.append(pred_state)
        target_filter.append(target_state)

    pred_filter = torch.stack(pred_filter)
    target_filter = torch.stack(target_filter)

    loss = (dt / tau) * torch.mean((pred_filter - target_filter) ** 2)

    return loss

# Post-processing step



In [45]:
def smooth(spikes, window, threshold):
    kernel = np.ones(window) / window
    fraction = np.convolve(spikes.astype(float), kernel, mode="same")
    return fraction > threshold

# Training

In [46]:
from sklearn.model_selection import train_test_split

random.seed(42)
torch.manual_seed(42)

items = [
    (filename, data)
    for filename, data in mel_data.items()
    if "labels" in data
]

recording_classes = []

for filename, data in items:
    recipe = recipe_lookup[os.path.basename(filename)]

    if recipe["event_present"]:
        recording_classes.append(recipe["event_class"])
    else:
        recording_classes.append("background_only")

train_items, validation_items = train_test_split(
    items,
    test_size=0.2,
    random_state=42,
    stratify=recording_classes
)

print("Training recordings:", len(train_items))
print("Validation recordings:", len(validation_items))

Training recordings: 640
Validation recordings: 160


In [47]:
train_mat = np.concatenate([d["log_mel"] for _, d in train_items], axis=1)
mu = train_mat.mean(axis=1, keepdims=True)
sd = train_mat.std(axis=1, keepdims=True) + 1e-6

In [48]:
random.seed(42)

event_chunks = []
background_chunks = []

chunk_size = 100

for filename, data in train_items:

    log_mel = data["log_mel"].T
    labels = data["labels"]

    for start in range(0, len(labels), chunk_size):

        end = min(start + chunk_size, len(labels))

        x_chunk = log_mel[start:end]
        y_chunk = labels[start:end]

        if np.any(y_chunk > 0):
            event_chunks.append((filename, x_chunk, y_chunk))
        else:
            background_chunks.append((filename, x_chunk, y_chunk))

sampled_background_chunks = random.sample(
    background_chunks,
    min(len(background_chunks), 3 * len(event_chunks))
)

balanced_chunks = event_chunks + sampled_background_chunks
random.shuffle(balanced_chunks)

def single_chunk(x_chunk, y_chunk):
    x = (x_chunk - mu.T) / sd.T
    y = (y_chunk > 0).astype(np.float32)
    x = torch.tensor(x, dtype=torch.float32, device=device)
    y = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1)
    return x, y

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def single_file(data):
    log_mel = data["log_mel"].T
    x = (log_mel - mu.T) / sd.T
    y = (data["labels"] > 0).astype(np.float32)
    x = torch.tensor(x, dtype=torch.float32, device=device)
    y = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1)
    return x, y

epochs = 5

for epoch in range(epochs):
    model.train()
    batch_size = 16
    random.shuffle(balanced_chunks)
    total_loss, n_batches = 0.0, 0

    for i in range(0, len(balanced_chunks), batch_size):
        batch = [c for c in balanced_chunks[i:i + batch_size] if len(c[2]) == 100]
        if not batch:
            continue

        xs = np.stack([(c[1] - mu.T) / sd.T for c in batch])
        ys = np.stack([(c[2] > 0).astype(np.float32) for c in batch])
        x = torch.tensor(xs, dtype=torch.float32, device=device).transpose(0, 1)
        target = torch.tensor(ys, dtype=torch.float32, device=device).transpose(0, 1).unsqueeze(-1)

        optimizer.zero_grad()
        prediction = model(x)
        loss = van_rossum_loss(prediction, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    avg_train_loss = total_loss / n_batches

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for filename, data in validation_items:
            x, target = single_file(data)
            prediction = model(x)
            loss = van_rossum_loss(prediction, target)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(validation_items)

    print(f"Epoch {epoch + 1}/{epochs} "f"Train Loss: {avg_train_loss:.6f} "f"Validation Loss: {avg_val_loss:.6f}")

Epoch 1/5 Train Loss: 0.277127 Validation Loss: 0.137984
Epoch 2/5 Train Loss: 0.226265 Validation Loss: 0.111720
Epoch 3/5 Train Loss: 0.195756 Validation Loss: 0.159780
Epoch 4/5 Train Loss: 0.178904 Validation Loss: 0.147603
Epoch 5/5 Train Loss: 0.174870 Validation Loss: 0.166887


# Eval

In [59]:
model.eval()

TP = 0
FP = 0
TN = 0
FN = 0

with torch.no_grad():
    for filename, data in validation_items:
        x, target = single_file(data)

        prediction = model(x)

        predicted = prediction.squeeze(-1) > 0
        actual = target.squeeze(-1) > 0

        TP += ((predicted == 1) & (actual == 1)).sum().item()
        FP += ((predicted == 1) & (actual == 0)).sum().item()
        TN += ((predicted == 0) & (actual == 0)).sum().item()
        FN += ((predicted == 0) & (actual == 1)).sum().item()

print("TP:", TP)
print("FP:", FP)
print("TN:", TN)
print("FN:", FN)

precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

TP: 3323
FP: 7985
TN: 135767
FN: 3005
Precision: 0.2938627520336984
Recall: 0.5251264222494862
F1: 0.3768428168997307


In [58]:
for window in [9, 15]:
    for threshold in [0.4, 0.5, 0.6]:
        masks = [smooth(p, window, threshold) for p in all_pred]
        aeer, f1, TP, FP, FN = event_eval(masks, all_true, xi=0)
        print(f"window={window:2d}  threshold={threshold}  AEER={aeer:.3f}  F1={f1:.3f}")

window= 9  threshold=0.4  AEER=0.392  F1=0.695
window= 9  threshold=0.5  AEER=0.408  F1=0.684
window= 9  threshold=0.6  AEER=0.400  F1=0.690
window=15  threshold=0.4  AEER=0.408  F1=0.693
window=15  threshold=0.5  AEER=0.400  F1=0.705
window=15  threshold=0.6  AEER=0.450  F1=0.645


# YAMNet

In [ ]:
!pip install tensorflow tensorflow-hub

yamnet_model = hub.load("https://tfhub.dev/google/yamnet/1")


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [ ]:
class_map_path = tf.keras.utils.get_file(
    'yamnet_class_map.csv',
    'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv'
)

class_map = pd.read_csv(class_map_path)

yamnet_event_classes = {
    "babycry": 20,
    "glassbreak": 435,
    "gunshot": 421
}

print(yamnet_event_classes)

{'babycry': 20, 'glassbreak': 435, 'gunshot': 421}


# Feedback

In [ ]:
adaptive_threshold = 1.0
threshold_step = 0.05
min_threshold = 0.7
max_threshold = 1.3
snn_uncertainty_threshold = 0.2
yamnet_confidence_threshold = 0.5